# Tutorial 06 — Evaluation Suite

Tests each evaluator with synthetic sequences:

1. **Sequence metrics** — perplexity, amino-acid recovery (AAR), diversity
2. **Developability** — liability scanner, TAP score
3. **Naturalness** (offline backends only — no model download needed)
4. **Full evaluation report** — JSON output

> Structure metrics (ABB2 refold, pLDDT) require `ImmuneBuilder` and are skipped here.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import torch, random, string
torch.manual_seed(42)
random.seed(42)

AA = list("ACDEFGHIKLMNPQRSTVWY")

def rand_seq(n):
    return "".join(random.choices(AA, k=n))

# Ground-truth and generated sequences
references = [(rand_seq(120), rand_seq(110)) for _ in range(8)]
generated  = [(rand_seq(120), rand_seq(110)) for _ in range(8)]

## 1. Perplexity

In [ ]:
from src.eval.sequence import compute_perplexity
from src.utils.tokenizer import AntibodyTokenizer
from src.models.decoder import AntibodyDecoder
from src.models.pooling import build_pooler
from src.models.projectors import build_projector
from src.models.abllava import AbLLaVA

tok = AntibodyTokenizer()

# Build tiny model
decoder   = AntibodyDecoder(vocab_size=tok.vocab_size, d_model=128, n_layers=2, n_heads=4, d_ff=256)
projector = build_projector('mlp', d_in=512, d_out=128)
model     = AbLLaVA(decoder=decoder, projector=projector, pooler_name='cdr_mean')
model.eval()

# Build synthetic batch for perplexity
B, N = 4, 230
batch = {
    'struct_emb':   torch.randn(B, N, 512),
    'cdr_spans':    torch.tensor([[[26,32],[50,57],[95,102],[143,153],[169,173],[208,216]]]*B, dtype=torch.long),
    'pad_mask':     torch.ones(B, N, dtype=torch.bool),
    'plddt':        torch.rand(B, N)*30+70,
    'seq_ids':      torch.randint(5, tok.vocab_size, (B, 50)),
    'seq_pad_mask': torch.ones(B, 50, dtype=torch.long),
}

ppl = compute_perplexity(model, [batch], device='cpu')
print(f"Perplexity (random model, expected ~25): {ppl:.2f}")

## 2. Amino-acid Recovery (AAR) and Diversity

In [ ]:
from src.eval.sequence import compute_recovery, compute_diversity, levenshtein, aa_kl

ref_heavies = [h for h, _ in references]
gen_heavies = [h for h, _ in generated]

aar = compute_recovery(ref_heavies, gen_heavies)
print(f"AAR (random vs random, expected ~0.05): {aar:.3f}")

div_gen = compute_diversity(gen_heavies)
div_ref = compute_diversity(ref_heavies)
print(f"Diversity (generated) : {div_gen:.3f}")
print(f"Diversity (references): {div_ref:.3f}")

# AA frequency KL divergence
kl = aa_kl(ref_heavies, gen_heavies)
print(f"AA-frequency KL(ref||gen): {kl:.4f}")

## 3. CDR-level Recovery

In [ ]:
# Approximate CDR slices for demo (IMGT 0-indexed positions)
cdr_slices = {
    'H-CDR1': (26, 32),
    'H-CDR2': (50, 57),
    'H-CDR3': (95, 102),
}

print("Per-CDR amino-acid recovery:")
for cdr, (s, e) in cdr_slices.items():
    ref_cdrs = [h[s:e] for h in ref_heavies if len(h) > e]
    gen_cdrs = [h[s:e] for h in gen_heavies if len(h) > e]
    n = min(len(ref_cdrs), len(gen_cdrs))
    if n == 0:
        continue
    aar_cdr = compute_recovery(ref_cdrs[:n], gen_cdrs[:n])
    print(f"  {cdr}: AAR={aar_cdr:.3f}")

## 4. Developability — Liability Scanner

In [ ]:
from src.eval.developability import LiabilityScanner, compute_developability

scanner = LiabilityScanner()

# A sequence with known liabilities
test_seqs = [
    "EVQLVESGGGLVQPGGSLRLSCAASGFTFSSYWMSWVRQAPGKGLEWVANIKQDGSEKYYVDSVKGRFTISRDNAKNSLYLQMNSLRAEDTAVYYCARDNWNDAFDIWGQGTLVTVSS",
    "NNNNDDDDSSSSCCCCC",   # asparagine-rich, many potential liabilities
    "".join(random.choices(AA, k=100)),
]

print("Liability scan results:")
for seq in test_seqs:
    report = scanner.scan(seq)
    liab_count = sum(1 for v in report.values() if v)
    print(f"  [{seq[:20]}...]  liabilities={liab_count}/10")
    for name, flag in report.items():
        if flag:
            print(f"    - {name}")

## 5. TAP score

In [ ]:
from src.eval.developability import tap_score

pairs = [(h, l) for h, l in generated[:4]]
scores = [tap_score(h, l) for h, l in pairs]
print("TAP scores (0=bad, 1=good):")
for (h, l), s in zip(pairs, scores):
    print(f"  heavy={h[:15]}... light={l[:15]}...  TAP={s:.3f}")

## 6. compute_developability — aggregated report

In [ ]:
import json

dev_report = compute_developability(pairs)
print(json.dumps(dev_report, indent=2))

## 7. Sequence metrics — full compute_sequence_metrics

In [ ]:
# Manual recreation of what evaluate.py does
report = {
    "perplexity": ppl,
    "aar_global":  aar,
    "diversity":   div_gen,
    "aa_kl":       kl,
    "developability": dev_report,
}
print("Final evaluation report:")
print(json.dumps(report, indent=2, default=str))

---
**Summary**: The evaluation suite produces a hierarchical JSON report covering sequence quality (perplexity, AAR, diversity), sequence naturalness (requires optional backends), and developability (liabilities, TAP, OASis). Structure metrics (refold pLDDT, RMSD) require `ImmuneBuilder` and run as an additional optional step.